# 03 — Cleaning

**Étape 5 — Nettoyage (fuseau, doublons, trous, unités, merge)**

Fusionner toutes les sources brutes (`data/raw/`) sur un index horaire commun
et produire le dataset `data/processed/dataset_v1.csv` documenté par le data dictionary
(README, section 8.2 du cahier de projet).


## Setup

In [13]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import data, features, models, backtest, metrics, plots

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (14, 4)


## Charger toutes les sources brutes (prix, fondamentaux, météo)

In [14]:
from pathlib import Path

In [15]:
RAW_DIR = Path("../data/raw")

PRICE_PATH = RAW_DIR / "entsoe_day_ahead_prices_fr.csv"
WEATHER_PATH = RAW_DIR / "openmeteo_weather_paris.csv"
FUNDAMENTALS_PATH = RAW_DIR / "rte_eco2mix_hourly_fr.csv"

for path in [PRICE_PATH, WEATHER_PATH, FUNDAMENTALS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier absent : {path}")

prices = pd.read_csv(
    PRICE_PATH,
    index_col="datetime",
)

weather = pd.read_csv(
    WEATHER_PATH,
    index_col="datetime",
)

fundamentals = pd.read_csv(
    FUNDAMENTALS_PATH,
    index_col="datetime",
)

print("Prix :", prices.shape)
print("Météo :", weather.shape)
print("Fondamentaux :", fundamentals.shape)

Prix : (32855, 1)
Météo : (32855, 3)
Fondamentaux : (32855, 13)


## Harmoniser le fuseau horaire (Europe/Paris) et gérer les changements d'heure (mars/octobre)

In [16]:
def normalize_utc_index(frame, name):
    frame = frame.copy()

    frame.index = pd.to_datetime(
        frame.index,
        utc=True,
        errors="raise",
    )

    frame.index.name = "datetime"
    frame = frame.sort_index()

    print(
        name,
        "| début :", frame.index.min(),
        "| fin :", frame.index.max(),
        "| fuseau :", frame.index.tz,
    )

    return frame


prices = normalize_utc_index(prices, "Prix")
weather = normalize_utc_index(weather, "Météo")
fundamentals = normalize_utc_index(
    fundamentals,
    "Fondamentaux",
)

print("\nDébut en heure française :")
print(prices.index.min().tz_convert("Europe/Paris"))

print("Fin en heure française :")
print(prices.index.max().tz_convert("Europe/Paris"))

Prix | début : 2021-12-31 23:00:00+00:00 | fin : 2025-09-30 21:00:00+00:00 | fuseau : UTC
Météo | début : 2021-12-31 23:00:00+00:00 | fin : 2025-09-30 21:00:00+00:00 | fuseau : UTC
Fondamentaux | début : 2021-12-31 23:00:00+00:00 | fin : 2025-09-30 21:00:00+00:00 | fuseau : UTC

Début en heure française :
2022-01-01 00:00:00+01:00
Fin en heure française :
2025-09-30 23:00:00+02:00


## Détecter et traiter les doublons

In [17]:
frames = {
    "prices": prices,
    "weather": weather,
    "fundamentals": fundamentals,
}

duplicate_report = {}

for name, frame in frames.items():
    duplicate_count = int(frame.index.duplicated().sum())
    duplicate_report[name] = duplicate_count

    print(f"{name} : {duplicate_count} doublon(s)")

    if duplicate_count > 0:
        display(frame.loc[frame.index.duplicated(keep=False)])

assert sum(duplicate_report.values()) == 0

prices : 0 doublon(s)
weather : 0 doublon(s)
fundamentals : 0 doublon(s)


## Détecter et documenter les valeurs manquantes (interpolation vs suppression, à justifier)

In [7]:
for name, frame in frames.items():
    print(f"\n{name}")
    print(frame.isna().sum())

print(
    "\nHeures fondamentales interpolées :",
    fundamentals["data_imputed"].sum(),
)

assert prices.isna().sum().sum() == 0
assert weather.isna().sum().sum() == 0
assert fundamentals.isna().sum().sum() == 0
assert fundamentals["data_imputed"].sum() == 3


prices
price_da    0
dtype: int64

weather
temperature_2m         0
wind_speed_10m         0
shortwave_radiation    0
dtype: int64

fundamentals
consommation     0
prevision_j1     0
fioul            0
charbon          0
gaz              0
nucleaire        0
eolien           0
solaire          0
hydraulique      0
pompage          0
bioenergies      0
ech_physiques    0
data_imputed     0
dtype: int64

Heures fondamentales interpolées : 3


## Vérifier les unités (MW vs GW vs MWh vs €/MWh vs €/tCO2)

In [18]:
fundamentals = fundamentals.rename(
    columns={
        "consommation": "demand_actual",
        "prevision_j1": "demand_forecast_j1",
        "fioul": "oil_generation",
        "charbon": "coal_generation",
        "gaz": "gas_generation",
        "nucleaire": "nuclear_generation",
        "eolien": "wind_generation",
        "solaire": "solar_generation",
        "hydraulique": "hydro_generation",
        "pompage": "pumping_consumption",
        "bioenergies": "bioenergy_generation",
        "ech_physiques": "net_import_mw",
        "data_imputed": "fundamentals_imputed",
    }
)

units = {
    "price_da": "EUR/MWh",
    "temperature_2m": "°C",
    "wind_speed_10m": "m/s",
    "shortwave_radiation": "W/m²",
    "demand_actual": "MW",
    "demand_forecast_j1": "MW",
    "oil_generation": "MW",
    "coal_generation": "MW",
    "gas_generation": "MW",
    "nuclear_generation": "MW",
    "wind_generation": "MW",
    "solar_generation": "MW",
    "hydro_generation": "MW",
    "pumping_consumption": "MW",
    "bioenergy_generation": "MW",
    "net_import_mw": "MW",
    "fundamentals_imputed": "flag 0/1",
}

pd.Series(units, name="unit").to_frame()

,unit
price_da,EUR/MWh
temperature_2m,°C
wind_speed_10m,m/s
shortwave_radiation,W/m²
demand_actual,MW
demand_forecast_j1,MW
oil_generation,MW
coal_generation,MW
gas_generation,MW
nuclear_generation,MW


## Merge final sur un index horaire unique

In [19]:
assert prices.index.equals(weather.index)
assert prices.index.equals(fundamentals.index)

dataset = (
    prices
    .join(
        weather,
        how="left",
        validate="one_to_one",
    )
    .join(
        fundamentals,
        how="left",
        validate="one_to_one",
    )
)

dataset.index.name = "datetime"

print("Dimensions finales :", dataset.shape)
print("Début :", dataset.index.min())
print("Fin :", dataset.index.max())

dataset.head()

Dimensions finales : (32855, 17)
Début : 2021-12-31 23:00:00+00:00
Fin : 2025-09-30 21:00:00+00:00


,price_da,temperature_2m,wind_speed_10m,shortwave_radiation,demand_actual,demand_forecast_j1,oil_generation,coal_generation,gas_generation,nuclear_generation,wind_generation,solar_generation,hydro_generation,pumping_consumption,bioenergy_generation,net_import_mw,fundamentals_imputed
datetime,,,,,,,,,,,,,,,,,
2021-12-31 23:00:00+00:00,89.06,9.4,3.111111,0.0,54315.0,50787.5,94.0,13.5,2826.0,40497.0,3187.5,0.0,9465.5,-264.0,1247.5,-2679.0,0
2022-01-01 00:00:00+00:00,78.48,10.7,2.500000,0.0,52173.5,49162.5,99.0,13.5,2693.0,38645.5,3270.0,0.0,9108.0,-65.0,1255.0,-2768.5,0
2022-01-01 01:00:00+00:00,85.16,9.3,2.638889,0.0,51527.5,47550.0,99.0,13.0,2687.0,38081.0,3146.0,0.0,8427.0,-602.5,1250.5,-1491.0,0
2022-01-01 02:00:00+00:00,50.00,8.8,2.833333,0.0,48845.5,44687.5,99.0,13.0,2694.0,37259.5,3273.5,0.0,6787.0,-1459.5,1247.5,-986.5,0
2022-01-01 03:00:00+00:00,37.67,8.2,2.944444,0.0,46646.0,43075.0,99.0,13.0,2699.5,36374.5,3481.5,0.0,6443.0,-2638.0,1253.5,-1001.0,0


## Sanity checks : bilans simples, ruptures de série (section 5)

In [20]:
expected_index = pd.date_range(
    start=dataset.index.min(),
    end=dataset.index.max(),
    freq="h",
    tz="UTC",
)

missing_hours = expected_index.difference(dataset.index)
extra_hours = dataset.index.difference(expected_index)

print("Doublons :", dataset.index.duplicated().sum())
print("Heures manquantes :", len(missing_hours))
print("Heures supplémentaires :", len(extra_hours))
print("Valeurs manquantes :", dataset.isna().sum().sum())
print("Heures interpolées :", dataset["fundamentals_imputed"].sum())

print("\nStatistiques prix :")
display(dataset["price_da"].describe())

print(
    "Prix négatifs :",
    (dataset["price_da"] < 0).sum(),
)

assert len(dataset) == 32855
assert dataset.index.is_monotonic_increasing
assert dataset.index.duplicated().sum() == 0
assert len(missing_hours) == 0
assert len(extra_hours) == 0
assert dataset.isna().sum().sum() == 0
assert dataset["fundamentals_imputed"].sum() == 3

Doublons : 0
Heures manquantes : 0
Heures supplémentaires : 0
Valeurs manquantes : 0
Heures interpolées : 3

Statistiques prix :


count    32855.000000
mean       127.020701
std        124.330870
min       -134.940000
25%         50.000000
50%         95.000000
75%        159.475000
max       2987.780000
Name: price_da, dtype: float64

Prix négatifs : 996


In [21]:
generation_columns = [
    "oil_generation",
    "coal_generation",
    "gas_generation",
    "nuclear_generation",
    "wind_generation",
    "solar_generation",
    "hydro_generation",
    "bioenergy_generation",
]

total_generation = dataset[generation_columns].sum(axis=1)

balance_gap = (
    total_generation
    + dataset["net_import_mw"]
    + dataset["pumping_consumption"]
    - dataset["demand_actual"]
)

print("Écart de bilan électrique (MW) :")
display(balance_gap.describe(percentiles=[0.01, 0.5, 0.99]))

Écart de bilan électrique (MW) :


count    32855.000000
mean        70.127119
std         24.677732
min        -37.500000
1%          18.500000
50%         68.500000
99%        128.000000
max        193.500000
dtype: float64

## Export vers `data/processed/dataset_v1.csv` + mise à jour du data dictionary

In [22]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROCESSED_DIR / "dataset_v1.csv"
DICTIONARY_PATH = PROCESSED_DIR / "data_dictionary_v1.csv"

dataset.to_csv(
    DATASET_PATH,
    index=True,
    date_format="%Y-%m-%d %H:%M:%S%z",
)

data_dictionary = pd.DataFrame(
    [
        ("datetime", "UTC timestamp", "index", "Clé horaire unique"),
        ("price_da", "EUR/MWh", "target", "Prix day-ahead France"),
        ("temperature_2m", "°C", "diagnostic_only", "Température réalisée à Paris"),
        ("wind_speed_10m", "m/s", "diagnostic_only", "Vent réalisé à Paris"),
        ("shortwave_radiation", "W/m²", "diagnostic_only", "Irradiance réalisée à Paris"),
        ("demand_actual", "MW", "diagnostic_only", "Consommation réalisée"),
        ("demand_forecast_j1", "MW", "candidate", "Prévision de demande J-1"),
        ("oil_generation", "MW", "diagnostic_only", "Production fioul réalisée"),
        ("coal_generation", "MW", "diagnostic_only", "Production charbon réalisée"),
        ("gas_generation", "MW", "diagnostic_only", "Production gaz réalisée"),
        ("nuclear_generation", "MW", "diagnostic_only", "Production nucléaire réalisée"),
        ("wind_generation", "MW", "diagnostic_only", "Production éolienne réalisée"),
        ("solar_generation", "MW", "diagnostic_only", "Production solaire réalisée"),
        ("hydro_generation", "MW", "diagnostic_only", "Production hydraulique réalisée"),
        ("pumping_consumption", "MW", "diagnostic_only", "Pompage STEP réalisé"),
        ("bioenergy_generation", "MW", "diagnostic_only", "Production bioénergies réalisée"),
        ("net_import_mw", "MW", "diagnostic_only", "Positif import, négatif export"),
        ("fundamentals_imputed", "0/1", "quality_flag", "Trois heures RTE interpolées"),
    ],
    columns=[
        "variable",
        "unit",
        "model_status",
        "description",
    ],
)

data_dictionary.to_csv(
    DICTIONARY_PATH,
    index=False,
)

print("Dataset :", DATASET_PATH)
print("Dictionnaire :", DICTIONARY_PATH)

Dataset : ../data/processed/dataset_v1.csv
Dictionnaire : ../data/processed/data_dictionary_v1.csv
